<a href="https://colab.research.google.com/github/alimoorreza/CS167-fall26-notes/blob/main/Day09_weighted_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS167: Day09
## Weighted k Nearest Neighbors

#### CS167: Machine Learning, Fall 2026


📜 [Syllabus](https://analytics.drake.edu/~reza/teaching/cs167_fall26/cs167_syllabus_fall26.pdf)

In [1]:
#run this cell if you're using Colab:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
path1 = '/content/drive/MyDrive/cs167_fall26/datasets/irisData.csv'
iris = pd.read_csv(path1)
iris.head(7)

,sepal length,sepal width,petal length,petal width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
5,5.4,3.9,1.7,0.4,Iris-setosa
6,4.6,3.4,1.4,0.3,Iris-setosa


## Are all neighbors created equal?

The way we've learned kNN so far, each neighbor gets an equal vote in the decision of what to predict.

Do we see any problems with this? If so, what?

<div>
<img src="https://analytics.drake.edu/~reza/teaching/cs167_sp25/notes/images/day04_wknn_motivation.png" width = 400/>
</div>

Should neighbors that are closer to the new instance get a larger share of the vote?

# Weighted k-NNN Intuition:

In weighted kNN, the nearest k points are given a weight, and the weights are grouped by the target variable. The class with the largest sum of weights will be the class that is predicted.

The intuition is to give more weight to the points that are nearby and less weight to the points that are farther away.
- distance-weighted voting

In w-kNN, we want to predict the target variable with the most weight, where the weight is defined by the inverse distance function.

## $w_{q,i} = \frac{1}{d(x_q, x_i)^2}$

> In English, you can read that as the __weight__ of a traning example is equal to 1 divided by the distance between the new instance and the traning example squared.

## A w-kNN Example: Step 1

Start by calculating the distance between the new example ('X'), and each of the other training examples:

<div>
<img src="https://analytics.drake.edu/~reza/teaching/cs167_sp25/notes/images/day04_wknn_ex.png" width=700/>
</div>

## A w-kNN Example: Step 2

Then, __calculate the weight__ of each training example using the inverse distance squared.

<div>
<img src="https://analytics.drake.edu/~reza/teaching/cs167_sp25/notes/images/day04_wknn_ex1.png" width=700/>
</div>

## A w-kNN Example: Step 3

Find the k closest neighbors--let's assume `k=3` for this example:
<div>
<img src="https://analytics.drake.edu/~reza/teaching/cs167_sp25/notes/images/day04_wknn_ex2.png" width=700/>
</div>

Then, sum the weights for each possible class:
- __orange__: $1$
- __blue__: $1/16 + 1/9 = 0.115$

### What would a __normal 3NN__ predict? Weighted 3NN?

## Let's write some code:

Let's use the sklearn's `weighted_knn` implementation.

In [3]:
#classic scikit-learn algorithm

#---------------- STEP 0 ----------------
# import libraries
import pandas
import sklearn
from sklearn.neighbors import KNeighborsClassifier

# load data
path = '/content/drive/MyDrive/cs167_fall26/datasets/irisData.csv'
iris_data = pandas.read_csv(path)
iris_data.head()

#---------------- STEP 1 ----------------
#1. load data
predictors   = ['sepal length', 'sepal width', 'petal length', 'petal width']
target       = 'species'
train_data   = iris_data[predictors]
train_labels = iris_data[target]
print('train_data shape:', train_data.shape, 'and train_labels shape:', train_labels.shape)
train_data.head()


train_data shape: (150, 4) and train_labels shape: (150,)


,sepal length,sepal width,petal length,petal width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [25]:
new_iris = {'sepal length': [7.2], 'sepal width':[2.5], 'petal length':5.1, 'petal width':[1.5]}
test_data = pandas.DataFrame(data=new_iris)
print('Our new test sample:')
test_data


Our new test sample:


,sepal length,sepal width,petal length,petal width
0,7.2,2.5,5.1,1.5


# scikit-learn algorithm for k-Nearest Neighbor

In [26]:
def knn_classifier(train_data, train_labels, test_data, k_value = 5):
  #---------------- STEP 2 ----------------
  #2. Create kNN classifier/regressor object
  from sklearn.neighbors import KNeighborsClassifier
  my_knn = KNeighborsClassifier(n_neighbors=k_value)


  #---------------- STEP 3 ----------------
  #3. Call fit (to train the classification/regression model)
  my_knn.fit(train_data, train_labels)


  #---------------- STEP 4 ----------------
  #4. Call predict to generate predictions
  predictions = my_knn.predict(test_data)
  print(f'Label of the new sample according to {k_value}-NN: {predictions[0]}')
  return my_knn, predictions



# scikit-learn algorithm for Weighted k-Nearest Neighbor

In [27]:
#classic scikit-learn algorithm

def weighted_knn_classifier(train_data, train_labels, test_data, k_value = 5):
  #---------------- STEP 2 ----------------
  #2. Create kNN classifier/regressor object
  from sklearn.neighbors import KNeighborsClassifier

  # use value 'distance' for weights parameter, by default it uses uniform weights for all data points
  my_knn = KNeighborsClassifier(n_neighbors=k_value, weights='distance')


  #---------------- STEP 3 ----------------
  #3. Call fit (to train the classification/regression model)
  my_knn.fit(train_data, train_labels)


  #---------------- STEP 4 ----------------
  #4. Call predict to generate predictions
  predictions = my_knn.predict(test_data)
  print(f'Label of the new sample according to {k_value}-NN: {predictions[0]}')
  return my_knn, predictions



In [28]:
# testing prediction on our new sample
my_knn, prediction = weighted_knn_classifier(train_data, train_labels, test_data, k_value = 5)
print('weighted kNN,   k=5:', prediction)

Label of the new sample according to 5-NN: Iris-versicolor
weighted kNN,   k=5: ['Iris-versicolor']


In [29]:
# Find the 5-nearest-neighbor training samples along with their distances to the test sample.

[dist, indx] = my_knn.kneighbors(test_data, 5)
dist, indx = dist[0], indx[0]
print('training indices are: ', indx)
print('distances are: ', dist)
print("-"*90)
print("from our current test_data the distances to the nearest five training samples are as follows:")
print("-"*90)
j = 0
for i in indx:
  train_sample = train_data.iloc[i][ ['sepal length', 'sepal width', 'petal length', 'petal width'] ]
  print(f"train_data:({train_sample['sepal length']}, {train_sample['sepal width']}, {train_sample['petal length']}, {train_sample['petal width']} ) with a distance of {dist[j]:.2f} and its label is:{train_labels.iloc[i]}")
  j = j + 1


training indices are:  [ 76  52  77  50 129]
distances are:  [0.59160798 0.7        0.74161985 0.83666003 0.8660254 ]
------------------------------------------------------------------------------------------
from our current test_data the distances to the nearest five training samples are as follows:
------------------------------------------------------------------------------------------
train_data:(6.8, 2.8, 4.8, 1.4 ) with a distance of 0.59 and its label is:Iris-versicolor
train_data:(6.9, 3.1, 4.9, 1.5 ) with a distance of 0.70 and its label is:Iris-versicolor
train_data:(6.7, 3.0, 5.0, 1.7 ) with a distance of 0.74 and its label is:Iris-versicolor
train_data:(7.0, 3.2, 4.7, 1.4 ) with a distance of 0.84 and its label is:Iris-versicolor
train_data:(7.2, 3.0, 5.8, 1.6 ) with a distance of 0.87 and its label is:Iris-virginica


In [30]:
# Calculate the weight of each training example using the inverse distance squared.

versicolor_weights = (1/dist[0]**2 + 1/dist[1]**2 + 1/dist[2]**2 + 1/dist[3]**2) # inverse square distances of all the Iris-versicolor predictions
virginica_weights = (1/dist[4]**2) # inverse square distances of the Iris-virginica prediction
print('versicolor total weight: ', versicolor_weights)
print('virginica total weight: ', virginica_weights)

versicolor total weight:  8.144712430426717
virginica total weight:  1.3333333333333328


### Dissecting why the new sample is predicted the way it is
Because the total weight associated with *versicolor* is 8.14, which is higher than the total weight of *virginica* (1.33), the model predicts the sample as `Iris-versicolor`.

## **Group Exercise#1:**

Normalize each of the predictor columns in the iris dataset `train_normalized`, and then try weighted kNN using the normalized data with a `k=5`.

>__Note__: you need a way to transform the new reading (the specimen) that you will make the prediction on so that the new one and the training data will all be on the same scale. How can you do that? Hint: Recall from last week’s lecture that we used `StandardScaler()` from scikit-learn for data normalization. [Here is the reference.](https://github.com/alimoorreza/CS167-fall26-notes/blob/main/Day08_Normalization.ipynb)

In [31]:
import numpy as np
from sklearn.preprocessing import StandardScaler

scaler                  = StandardScaler()

scaler.fit(train_data)                                  # computes the mean and std to be used for later scaling
train_data_normalized   = scaler.transform(train_data)  # performs standardization by centering and scaling on the training features
test_data_normalized    = scaler.transform(test_data)   # performs standardization by centering and scaling on the testing features
#train_data_normalized
print('mean:',scaler.mean_)
print('std:', np.sqrt(scaler.var_))
print(train_data_normalized[0:5]) # first 5 training samples with normalized values

mean: [5.84333333 3.054      3.75866667 1.19866667]
std: [0.82530129 0.43214658 1.75852918 0.76061262]
[[-0.90068117  1.03205722 -1.3412724  -1.31297673]
 [-1.14301691 -0.1249576  -1.3412724  -1.31297673]
 [-1.38535265  0.33784833 -1.39813811 -1.31297673]
 [-1.50652052  0.10644536 -1.2844067  -1.31297673]
 [-1.02184904  1.26346019 -1.3412724  -1.31297673]]


In [33]:
print(test_data_normalized) # test sample with normalized values

[[ 1.64384411 -1.28197243  0.76275864  0.39617188]]


### **Step#1**:
Find k-NN prediction for the normalized data uisng `k=5` and weighted kNN algorithm.

In [39]:
# testing prediction on our new sample
# your code here
# ...
# ...
# ...

Label of the new sample according to 5-NN: Iris-versicolor


### **Step#2**:
Dissect why the new sample is predicted the way it is.

In [40]:
# Find the 5-nearest-neighbor training samples along with their distances to the test sample.
# your code here
# ...
# ...
# ...


training indices are:  [108  76 130  72  54]
distances are:  [0.82526317 0.87362518 1.06798207 1.09642542 1.13232932]
------------------------------------------------------------------------------------------
from our current test_data the distances to the nearest five training samples are as follows:
------------------------------------------------------------------------------------------
train_data:(6.7, 2.5, 5.8, 1.8 ) with a distance of 0.83 and its label is:Iris-virginica
train_data:(6.8, 2.8, 4.8, 1.4 ) with a distance of 0.87 and its label is:Iris-versicolor
train_data:(7.4, 2.8, 6.1, 1.9 ) with a distance of 1.07 and its label is:Iris-virginica
train_data:(6.3, 2.5, 4.9, 1.5 ) with a distance of 1.10 and its label is:Iris-versicolor
train_data:(6.5, 2.8, 4.6, 1.5 ) with a distance of 1.13 and its label is:Iris-versicolor


In [41]:
# Calculate the weight of each training example using the inverse distance squared.
# your code here
# ...
# ...
# ...

versicolor total weight:  2.9220083754060275
virginica total weight:  2.345043453659717


### **What is your final interpretation based on the above analysis?**
What does the model predict the sample as `Iris-virginica` or `Iris-versicolor`? Interpret the result based on the weighted distances.

### your answer:
???

???

???

## **Group Exercise#2:**
Repeat your k-NN prediction code for the normalized data using other values of `k`.
- Does the value of k change the predictions?
    - compare using `k=3`, and `k=5` on each method (normalized and non-normalized), (weighted and unweighted)

In [ ]:
my_knn, prediction = knn_classifier(train_data, train_labels, test_data, k_value = 3)
print('unweighted kNN, k=3:', prediction)
my_knn, prediction = knn_classifier(train_data, train_labels, test_data, k_value = 5)
print('unweighted kNN, k=5:', prediction)
my_knn, prediction = weighted_knn_classifier(train_data, train_labels, test_data, k_value = 3)
print('weighted kNN,   k=3:', prediction)

In [ ]:
my_knn, prediction = knn_classifier(train_data_normalized, train_labels, test_data_normalized, k_value = 3)
print('unweighted kNN, k=3:', prediction)
my_knn, prediction = knn_classifier(train_data_normalized, train_labels, test_data_normalized, k_value = 5)
print('unweighted kNN, k=5:', prediction)
my_knn, prediction = weighted_knn_classifier(train_data_normalized, train_labels, test_data_normalized, k_value = 3)
print('weighted kNN,   k=3:', prediction)


## Use these tables to keep track of your predictions:
### `k=3`
|                    | **not normalized** | **normalized** |
|--------------------|--------------------|----------------|
| **kNN** |          |              |
| **weighted kNN**   |          |               |

### `k=5`

|                    | **not normalized** | **normalized** |
|--------------------|--------------------|----------------|
| **kNN** |                    |                |
| **weighted kNN**   |                    |                |